# RQ2 and RQ3 - Fisher's r-to-z comparisons
RQ2 compares Llama 3.1-70B Base to Llama 3.3-70B Instruct.
RQ3 compares Llama 3.3-70B Instruct to Qwen 2.5-72B Instruct.
Both use Fisher's r-to-z transformation to test for significant differences in the model-human correlation.

## Load data

In [ ]:
"""Load the merged data using the project's data_loading module."""
from data_loading import load_data

df = load_data()
df.head()


## Helper: Fisher's r-to-z test for two independent correlations

In [ ]:
import numpy as np
from scipy.stats import pearsonr, norm

def fisher_z_test(r1, n1, r2, n2):
    z1 = 0.5 * np.log((1 + r1) / (1 - r1))
    z2 = 0.5 * np.log((1 + r2) / (1 - r2))
    se = np.sqrt(1.0 / (n1 - 3) + 1.0 / (n2 - 3))
    z  = (z1 - z2) / se
    p  = 2 * (1 - norm.cdf(abs(z)))
    return z, p


## RQ2: Llama Base vs. Llama Instruct

In [ ]:
import pandas as pd

rq2_pairs = [
    ("Explicit", "llama_base_response",  "llama_instruct_response"),
    ("Implicit", "llama_base_expected",  "llama_instruct_expected"),
]

rq2_rows = []
for label, base_col, inst_col in rq2_pairs:
    base = df[["human", base_col]].dropna()
    inst = df[["human", inst_col]].dropna()
    r_base, _ = pearsonr(base["human"], base[base_col])
    r_inst, _ = pearsonr(inst["human"], inst[inst_col])
    z, p = fisher_z_test(r_base, len(base), r_inst, len(inst))
    rq2_rows.append({
        "Measure": label,
        "r (Base)": r_base, "N (Base)": len(base),
        "r (Instruct)": r_inst, "N (Instruct)": len(inst),
        "Difference": r_inst - r_base, "z": z, "p": p,
    })

pd.DataFrame(rq2_rows)


## RQ3: Llama Instruct vs. Qwen

In [ ]:
rq3_pairs = [
    ("Explicit", "llama_instruct_response", "qwen_response"),
    ("Implicit", "llama_instruct_expected", "qwen_expected"),
]

rq3_rows = []
for label, ll_col, qw_col in rq3_pairs:
    ll = df[["human", ll_col]].dropna()
    qw = df[["human", qw_col]].dropna()
    r_ll, _ = pearsonr(ll["human"], ll[ll_col])
    r_qw, _ = pearsonr(qw["human"], qw[qw_col])
    z, p = fisher_z_test(r_ll, len(ll), r_qw, len(qw))
    rq3_rows.append({
        "Measure": label,
        "r (Llama Instruct)": r_ll, "N (Llama)": len(ll),
        "r (Qwen)": r_qw, "N (Qwen)": len(qw),
        "Difference": r_qw - r_ll, "z": z, "p": p,
    })

pd.DataFrame(rq3_rows)


## Build the RQ2 table image (booktabs-style PNG)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

def render_table(columns, rows, out_name, col_x=None):
    mpl.rcParams["font.family"] = "DejaVu Serif"
    fig, ax = plt.subplots(figsize=(11, 2.0), dpi=220)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    if col_x is None:
        col_x = [0.20, 0.42, 0.55, 0.70, 0.82, 0.92]
    header_y = 0.78
    row_ys   = [0.55, 0.35]
    ax.hlines(0.92, 0.03, 0.97, colors="black", linewidth=2.0)
    ax.hlines(0.68, 0.03, 0.97, colors="black", linewidth=1.0)
    ax.hlines(0.22, 0.03, 0.97, colors="black", linewidth=2.0)
    aligns = ["left"] + ["center"] * 5
    for i, label in enumerate(columns):
        x = col_x[i] if aligns[i] == "center" else 0.05
        ax.text(x, header_y, label, ha=aligns[i], va="center",
                fontsize=12, fontweight="bold")
    for ri, row in enumerate(rows):
        for ci, val in enumerate(row):
            align = "left" if ci == 0 else "center"
            x = col_x[ci] if align == "center" else 0.05
            ax.text(x, row_ys[ri], val, ha=align, va="center", fontsize=12)
    plt.tight_layout(pad=0.4)
    plt.savefig(out_name, dpi=300, bbox_inches="tight", pad_inches=0.15)
    plt.show()

render_table(
    ["Measure", "r (Base)", "r (Instruct)", "Difference", "z", "p"],
    [["Explicit response",       ".06", ".51", "+.46", "-71.77", "< .001"],
     ["Implicit expected score", ".20", ".53", "+.33", "-53.74", "< .001"]],
    out_name="table2_rq2_fisher.png",
)


## Build the RQ3 table image

In [ ]:
render_table(
    ["Measure", "r (Llama Instruct)", "r (Qwen)", "Difference", "z", "p"],
    [["Explicit response",       ".51", ".53", "+.02",  "-3.41", "< .001"],
     ["Implicit expected score", ".53", ".60", "+.07", "-14.66", "< .001"]],
    out_name="table4_rq3_fisher.png",
    col_x=[0.20, 0.46, 0.61, 0.73, 0.84, 0.93],
)
